In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install alpaca-py xgboost scikit-learn pandas matplotlib

# 2. IMPORT LIBRARIES
import os
import pandas as pd
import matplotlib.pyplot as plt

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

# 3. CONFIGURE API AND PARAMETERS
# Note: Input your own free Alpaca Paper Trading keys here to run the notebook.
API_KEY = "YOUR_ALPACA_API_KEY"
API_SECRET = "YOUR_ALPACA_SECRET_KEY"

SYMBOL = "NVDA"
START_DATE = "2022-01-01"
END_DATE = "2026-11-13"

client = StockHistoricalDataClient(API_KEY, API_SECRET)

request_params = StockBarsRequest(
    symbol_or_symbols=SYMBOL,
    timeframe=TimeFrame.Day,
    start=START_DATE,
    end=END_DATE,
)

# 4. INGEST AND PREPARE DATA
bars = client.get_stock_bars(request_params)
df = bars.df.reset_index()
df = df[df["symbol"] == SYMBOL]

# Feature Engineering
df["return"] = df["close"].pct_change()
df["volatility"] = df["return"].rolling(5).std()
df["ma_5"] = df["close"].rolling(5).mean()
df["ma_10"] = df["close"].rolling(10).mean()
df["ma_20"] = df["close"].rolling(20).mean()

# Define Target Variable
df["target"] = df["close"].shift(-1)
df = df.dropna()

features = [
    "close",
    "volume",
    "volatility",
    "ma_5",
    "ma_10",
    "ma_20",
]

# 5. SPLIT DATA
X = df[features]
y = df["target"]

# 80/20 Chronological Time-Series Split
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# 6. TRAIN MODELS AND EVALUATE

# Random Forest
rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_preds)
print(f"Random Forest MAE: ${rf_mae:.2f}")

# XGBoost
xgb = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_preds)
print(f"XGBoost MAE: ${xgb_mae:.2f}")

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_preds)
print(f"Linear Regression MAE: ${lr_mae:.2f}")

# 7. PLOT RESULTS
plt.figure(figsize=(14, 7))
plt.plot(y_test.values, label="Actual", linewidth=2)
plt.plot(rf_preds, label="Random Forest")
plt.plot(xgb_preds, label="XGBoost")
plt.plot(lr_preds, label="Linear Regression")

plt.title("NVDA Price Prediction Model Comparison")
plt.xlabel("Days (Test Period)")
plt.ylabel("Price (USD)")
plt.legend()
plt.show()